# Demo 02 - Weather Data

Notebook này dùng để demo dữ liệu thời tiết theo giờ. Ưu tiên đọc `s3a://silver/hourly_weather/`; nếu chưa có Silver thì fallback về `s3a://bronze/weather/`.

## 4 phần nên chụp vào slide

1. Dataset overview
2. Schema / thuộc tính chính
3. Sample records
4. Weather quality summary

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebooks.utils.spark_session import get_spark, path_exists
from pyspark.sql.functions import col, count, countDistinct, max as spark_max, min as spark_min, sum as spark_sum, when

spark = get_spark("MetroPulse Demo - Weather Data")
spark.sparkContext.setLogLevel("WARN")
spark.conf.get("spark.sql.session.timeZone")

## Load weather dataset

In [ ]:
silver_weather_path = "s3a://silver/hourly_weather/"
bronze_weather_path = "s3a://bronze/weather/"

if path_exists(spark, silver_weather_path):
    weather_path = silver_weather_path
elif path_exists(spark, bronze_weather_path):
    weather_path = bronze_weather_path
else:
    raise FileNotFoundError(
        "Không tìm thấy weather data trên MinIO. Hãy chạy: make start && make weather && make bronze && make silver"
    )

weather = spark.read.parquet(weather_path).cache()
weather_path

## 1. Dataset overview

In [ ]:
overview = spark.createDataFrame(
    [
        ("Source", "Open-Meteo API"),
        ("Scope", "NYC area, historical hourly weather"),
        ("Expected grain", "1 row per hour"),
        ("Expected records", "17,542 hourly records"),
        ("Current demo path", weather_path),
    ],
    ["property", "value"],
)
overview.show(truncate=False)

## 2. Các thuộc tính chính

In [ ]:
attributes = spark.createDataFrame(
    [
        ("weather_time / weather_hour", "timestamp", "Thời điểm ghi nhận thời tiết"),
        ("temperature_2m_f", "double", "Nhiệt độ Fahrenheit"),
        ("precipitation_mm", "double", "Lượng mưa"),
        ("windspeed_10m", "double", "Tốc độ gió"),
        ("relative_humidity_2m", "double", "Độ ẩm tương đối"),
    ],
    ["column", "type", "meaning"],
)
attributes.show(truncate=False)
weather.printSchema()

## 3. Sample records

In [ ]:
preferred_columns = [
    "weather_hour",
    "weather_time",
    "temperature_2m_f",
    "precipitation_mm",
    "windspeed_10m",
    "relative_humidity_2m",
]
available_columns = [column_name for column_name in preferred_columns if column_name in weather.columns]

weather.select(*available_columns).show(5, truncate=False)

## 4. Weather quality summary

In [ ]:
time_col = "weather_hour" if "weather_hour" in weather.columns else "weather_time"

summary = weather.agg(
    count("*").alias("row_count"),
    countDistinct(time_col).alias("distinct_hours"),
    spark_min(time_col).alias("min_time"),
    spark_max(time_col).alias("max_time"),
)
summary.show(truncate=False)

quality_columns = [column_name for column_name in ["temperature_2m_f", "precipitation_mm", "windspeed_10m", "relative_humidity_2m"] if column_name in weather.columns]
if quality_columns:
    weather.agg(
        *[spark_sum(when(col(column_name).isNull(), 1).otherwise(0)).alias(f"null_{column_name}") for column_name in quality_columns]
    ).show(truncate=False)